# Modularized Script: Tour Planning
---
For dedicated customer use cases. Results in [this folder](https://drive.google.com/drive/folders/1UHngYb9uXdRfO_0N002zc2Rz0xnk3xbX?usp=drive_link)

In [1]:
'''
Directory structure on Google Drive:

MyDrive/Colab Notebooks/colab_modules_example/
*   module_upload.py
*   module_clean.py
*   module_geocoding.py
*   module_vrp
*   module_results
*   module_map
*   module_column_mapping
*   module_export
*   module_client_configuration

'''

# @title Initialization and loading of modules
from google.colab import drive
import sys, os, importlib

# 0. Initial Loading of parameters

# From colab secretes
from google.colab import userdata
api_key = userdata.get('apiKey_HERE')
# Define Planning Date
base_date_str = "2025-12-15" # @param {"type":"date"}

# Specify the path for the output folder for the planning day
client_name = "Kemmler" # @param ["Stark","Kemmler","laminatDepot_MultiBranch","Wigger","Obi_Buchholz","default"]

# Specify the path for the output folder for the planning day
product_category = "Ohne Modifikation" # '''@param ["Ohne Modifikation","bex Kurier","bex Tour"]'''

# Conditional logic for setting paths
if client_name == "Stark":
    depots_path = '/content/drive/MyDrive/Colab Notebooks/stark/depots/Depots_geocoded.xlsx'
    output_folder_path = '/content/drive/MyDrive/Colab Notebooks/stark/' + base_date_str
    sheet_name = 'Sheet1'
elif client_name == "Kemmler":
    depots_path = '/content/drive/MyDrive/Colab Notebooks/kemmler/depots/Depots_geocoded.xlsx'
    output_folder_path = '/content/drive/MyDrive/Colab Notebooks/kemmler/' + base_date_str
    sheet_name = 'Touren - BEX'
elif client_name == "laminatDepot_MultiBranch":
    depots_path = '/content/drive/MyDrive/Colab Notebooks/laminatdepot/depots/Depots_geocoded.xlsx'
    output_folder_path = '/content/drive/MyDrive/Colab Notebooks/laminatdepot/' + base_date_str
    sheet_name = 'input'
elif client_name == "Wigger":
    depots_path = '/content/drive/MyDrive/Colab Notebooks/Wigger/depots/Depots_geocoded.xlsx'
    output_folder_path = '/content/drive/MyDrive/Colab Notebooks/Wigger/' + base_date_str
    sheet_name = 'Touren - BEX'
elif client_name == "Obi_Buchholz":
    depots_path = '/content/drive/MyDrive/Colab Notebooks/Obi_Buchholz/depots/Depots_geocoded.xlsx'
    output_folder_path = '/content/drive/MyDrive/Colab Notebooks/Obi_Buchholz/' + base_date_str
    sheet_name = 'Touren - BEX'
else:
    # Default paths for other clients (e.g., Stark)
    depots_path = '/content/drive/MyDrive/Colab Notebooks/generic_tourplanning/depots/Depots_geocoded.xlsx'
    output_folder_path = '/content/drive/MyDrive/Colab Notebooks/generic_tourplanning/' + base_date_str
    sheet_name = 'Sheet1'

# 1. Mount Drive for Drive access
drive.mount('/content/drive')  # can skip if only widget upload needed

# 2. Add module folder to path
module_path = '/content/drive/MyDrive/Colab Notebooks/tourplanning_modules'
if module_path not in sys.path:
    sys.path.insert(0, module_path)

# 3. Import modules
import module_upload, module_clean, module_geocoding, module_vrp, module_results, module_map, module_column_mapping, module_export, module_client_configuration

# 4. Reload modules after any change
importlib.reload(module_upload)
importlib.reload(module_clean)
importlib.reload(module_geocoding)
importlib.reload(module_vrp)
importlib.reload(module_results)
importlib.reload(module_map)
importlib.reload(module_column_mapping)
importlib.reload(module_export)
importlib.reload(module_client_configuration)

Mounted at /content/drive


<module 'module_client_configuration' from '/content/drive/MyDrive/Colab Notebooks/tourplanning_modules/module_client_configuration.py'>

In [2]:
# @title Upload Template File

# 5. Upload file via widget and load DataFrame
df_raw = module_upload.upload_excel_via_widget(sheet_name)
print("Data successfully loaded as df_raw")
print("Raw data shape (rows, columns):", df_raw.shape)

Saving Kemmler - Auftragsspeicher - Touren - 2025-12-12T174021.807.xlsx to Kemmler - Auftragsspeicher - Touren - 2025-12-12T174021.807.xlsx
Data successfully loaded as df_raw
Raw data shape (rows, columns): (3145, 33)


In [3]:
# @title Prepare data for further processing and perform filtering

# 6. Clean data
df_clean = module_clean.clean_and_process_data(df_raw, base_date_str,output_folder_path,client_name)
print("Data successfully prepared for further processing as df_clean")
print("New output folder created.")
print("Cleaned data shape (rows, columns):", df_clean.shape)

Folder created at /content/drive/MyDrive/Colab Notebooks/kemmler/2025-12-15
Data successfully prepared for further processing as df_clean
New output folder created.
Cleaned data shape (rows, columns): (13, 40)


In [4]:
# @title Perform Geocoding for input data

# 7. Perform Geocoding for input data

df_geocoded = module_geocoding.geocoding_cleaned_data(df_clean, api_key,client_name)
print("Data is successfully enriched with geocodig data and available as df_geocoded")
print("Geocoden data shape (rows, columns):", df_geocoded.shape)

Data is successfully enriched with geocodig data and available as df_geocoded
Geocoden data shape (rows, columns): (13, 62)


In [5]:
# @title Perform Vehicle Routing Planning

# 8. Perform Vehicle Routing Planning

# From colab secretes
from google.colab import userdata
api_key = userdata.get('apiKey_HERE')

vrp_problem_statement_json = module_vrp.vrp_problem_definition(df_geocoded, api_key, depots_path, base_date_str, product_category,client_name)

import math

def replace_nan_with_none_recursive(obj):
    """Recursively replaces NaN and Inf float values in a nested dict/list with None."""
    if isinstance(obj, dict):
        return {k: replace_nan_with_none_recursive(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [replace_nan_with_none_recursive(elem) for elem in obj]
    elif isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)):
        return None
    else:
        return obj

# Clean the problem statement JSON before execution
vrp_problem_statement_json_cleaned = replace_nan_with_none_recursive(vrp_problem_statement_json)

vrp_response_json = module_vrp.vrp_problem_execution(vrp_problem_statement_json_cleaned,api_key)


Depots DataFrame loaded with 33 rows.
Filtered depots DataFrame with 1 rows.
Generated 3 fleet types.
Generated 13 jobs.
Dynamic JSON object created.
Request for VRP_Execution to HERE was successful.


In [6]:
# @title Merges the VRP response JSON with the geocoded DataFrame and provide as csv-file
# 9. Merge results

# Merges the VRP response JSON with the geocoded DataFrame with a leftjoin
merged_df, df_unassigned_jobs = module_results.merge_results(vrp_response_json, df_geocoded,output_folder_path,depots_path)
print("Data successfully merged as merged_df")
print("Merged data shape (rows, columns):", merged_df.shape)

# Saves the result to a CSV file.
merged_df.to_csv(output_folder_path+'/' + base_date_str + '.csv')

# Create a dedicated bexOS csv file
merged_df_bexOS = module_export.create_bexos_import_csv(merged_df)

# Save the dedicated bexOS export to a CSV file
#merged_df_bexOS.to_csv(base_date_str + '_import_to_bexOS.csv', index=False, sep=",", encoding="utf-8")
merged_df_bexOS.to_csv(output_folder_path+'/' + base_date_str + '_import_to_bexOS.csv', index=False, sep=",", encoding="utf-8")
print("Merged data saved to file, incl. bexOS import file")

Depots DataFrame loaded with 33 rows.
Data successfully merged as merged_df
Merged data shape (rows, columns): (31, 84)
Merged data saved to file, incl. bexOS import file


In [7]:
# @title Create a map with all tour plans
# 10. Create a `tour plan as map` and name how many unassigend orders there are.

map = module_map.create_map(merged_df, df_geocoded, df_unassigned_jobs, base_date_str)
print("Map successfully created")

# Save the map
map.save(output_folder_path+'/' + base_date_str + '_tours_map.html')
print("Map successfully saved to file")

Map successfully created
Map successfully saved to file


In [8]:
# @title Show the used planning parameters
# 11. Show the configuration parameters used by selection of client_name
import pandas as pd

config = module_client_configuration.CONFIG.get(client_name)

for client_config, client_config_group in config.items():
    # Convert the client's configuration to a DataFrame
    df = pd.json_normalize(client_config_group, sep='.')

    # Transpose the DataFrame for better readability
    df_transposed = df.transpose()

    # Display the client's configuration as a table with a title
    print(f"# Configuration for {client_name}:")
    print(f"## Configuration of {client_name}-{client_config}:")
    display(df_transposed)
    print("\n") # Add some space between tables

# Configuration for Kemmler:
## Configuration of Kemmler-order:


,0
timewindow,True




# Configuration for Kemmler:
## Configuration of Kemmler-break_duration:


,0
fullDay,2100
halfDay,900




# Configuration for Kemmler:
## Configuration of Kemmler-break_times:


...

,0
dedicatedVehicles.speedFactor,0.75
dedicatedVehicles.costs.fixed.fullDay,defined in depot_file
dedicatedVehicles.costs.fixed.halfDay,defined in depot_file
dedicatedVehicles.costs.distance.fullDay,0.001
dedicatedVehicles.costs.distance.halfDay,0.0015
dedicatedVehicles.costs.time.fullDay,0.014
dedicatedVehicles.costs.time.halfDay,0.02
dedicatedVehicles.shifts.start_location_depot,True
dedicatedVehicles.shifts.end_location_depot,False
dedicatedVehicles.limits.maxDistance.fullDay,350000




# Configuration for Kemmler:
## Configuration of Kemmler-job:


,0
pickup_duration,900
delivery_duration,900




# Configuration for Kemmler:
## Configuration of Kemmler-advancedObjectives:


,0,1,2,3,4
0,{'type': 'minimizeUnassigned'},{'type': 'minimizeTours'},{'type': 'minimizeCost'},"{'type': 'balanceDuration', 'options.threshold...",{'type': 'maximizeTerritoryJobs'}
